# Integration | movement and Pynapple
-----------------------------------------

## Setup and Configuration

In [ ]:
##### Imports ##########################################################################

#==== Generic Imports ================
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

#==== Movement Imports ===============
import movement.kinematics

#==== DataConduit Imports (Core) =====
from data_conduit.datastructures import slice_stream, slice_stream_per_trial
from data_conduit.integrations.DLC.pose import pose_to_movement     # Combine aligned pose and confidence in movement format.

#==== DataConduit Imports (Unique) ===
from movement_figures.data_template.loading import build_datastructure
from movement_figures.misc import visual_streams_check, configure_simplified_trial_df
from data_conduit.refactor_qc import (
    TRAINING_FIRST_MID_LAST_DETAIL,
    training_spec,
    filter_trials,
)

#==== Imports for This Figure =========
import pynapple as nap

#==== Time-Series Annotations =========
from collections.abc import Callable
from movement_figures.timeseries_template.annotations import (
    AnnotationResult,
    annotate_qc_timeseries,
)

### Notebook Settings ######################################################

pd.set_option('display.max_columns', None)                         # Show all trial columns during inspection.
pd.set_option('display.max_rows', None)


In [ ]:
##### Setup Parameters #################################################################

# === 1| Select Mice, Days and Recording Folders Before Reading Files =============
ROOT = Path("/media/sepi/Elements/PathIntegrationProtocol/BonsaiOutput/Training")

LEVEL_SELECTORS = {
    'l0_selector': 'MR_M01569515', 
    'l1_selector': 'Day21'
    }                                                                                       # Example choices belong here: l0_selector and l1_selector.
# INCLUDE_SESSIONS = ('2026-06-21T093349Z',)                                                # Optional list of recording folder names; None keeps selected folders.
# STREAMS = ('trials', 'events', 'video', 'dlc')                                            # These figures require events/trials and aligned pose; other sources remain available.


#=== 2| Set the trial filtering parameters for the experiment protocol ============
spec = training_spec(TRAINING_FIRST_MID_LAST_DETAIL)                                        # Flattens `{training_detail}` table into per-session spec. 

# included_sessions = training_session_names(spec)                                          # Returns a list of session folder names that match the spec.
#   Not used here, as sessions have been filtered in another way. If other sessions 
#   not in the above spec are used, then applying the `filter_trials` function with 
#   the above spec could result in incorrect filtering.


In [ ]:
##### Load DataStructure ###############################################################

#=== Load data ========================================================================

fig_ds_obj = build_datastructure(
    root               = ROOT,                                                                 # Directory above the named hierarchy levels.
    level_names        = ("mouseID", "day"),                                                   # Lab layout: root / mouseID / day / session.
    streams            = ("trials", "session_settings", "video", "dlc"),                       # Data streams to be included in the datastructure.
    include            = None,                                                                 # Keep these session folder names; None keeps all.
    exclude            = None,                                                                 # Alternatively omit these session folder names.
    raw_events         = True,                                                                 # Convenience alias for adding "events" to the requested sources.
    device_yaml        = "./device.yml",                                                       # Lab Nosepoke board register definitions.
    soundcard_yaml     = "./soundcard.yml",                                                    # Lab SoundCard register definitions.
    nosepoke_count     = 18,                                                                   # Port count used by the existing Q_C parser.
    trial_start_buffer = 0.0,                                                                  # Preserve the existing lab parser setting.
    dlc_file_format     = "auto",                                                              # Existing reader prefers CSV, otherwise HDF5.
    trial_spec = None,                                                                         # Optional experiment rules; None keeps the existing Q_C specification. 
    #    Not used here, as sessions are already selected. 
    #    If session is not one in the spec, then subsequent use of 
    #   `filter_trials` to apply correct experiment protocol rules could be incorrect.
    
    **LEVEL_SELECTORS,
)


#=== Select and load the streams of interest from datastructure. ======================
fig_ds_obj.select()
streams = fig_ds_obj.load()                                                                    # Load is required to access the selected streams.

#=== Create convenient compact trial dataframe without some unnecessary columns. ======
streams['compact_trials'] = configure_simplified_trial_df(              
    trial_df = streams['trials'],
    modify_in_place = False,                                                                   # Whether to modify the trial dataframe in place or return a new dataframe. 
)
'''More useful for visual inspection and analysis. 
Check the misc.py file for the default columns to keep and drop.'''


#=== Apply Trial Filtering Protocol to construct new stream objects ===================

# Full filtered trials
streams['filtered_trials'] = filter_trials(
    trials = streams['trials'],
    spec = spec,
    # session_column = 'session', mouse_column = 'mouseID', trial_index_column = 'trial_index', led_column = 'LED', report = True,
    #                                                            # Leave params untouched for now, let the function use its defaults.
)

# Filtered compact trials 
streams['filtered_compact_trials'] = filter_trials(
    trials = streams['compact_trials'],
    spec = spec,
    # session_column = 'session', mouse_column = 'mouseID', trial_index_column = 'trial_index', led_column = 'LED', report = True,
    #                                                            # Leave params untouched for now, let the function use its defaults.
)

#=== Merge Pose & Confidence Streams ==================================================
streams['dlc:movement'] = pose_to_movement(
    slice_stream(streams["dlc:position"],   
                #selectors={"session": session_id}
    ),
    slice_stream(streams["dlc:confidence"], 
                 #selectors={"session": session_id}
    ),
)

#=== Visual streams check =============================================================
visual_streams_check(
    loaded_ds_object = streams,                                                                # Loaded dataset object containing all streams of interest.
    show_stream_shapes = True,                                                                 # Whether to display the shapes of the streams.
    show_stream_dtypes = True,                                                                 # Whether to display the data types of the streams.
    display_streams = False                                                                    # Whether to display the actual stream data as a table.
)



## Select Trials to Show

In [ ]:
##### Trial Selection ##################################################################

#==== Extract Trial Range ============
selected_trials = slice_stream(streams['filtered_trials'][:8])              # Select the first n trials

#==== Extract Recording Identifier ===
sessionID = selected_trials['session'].unique().item()                      # Extract unique sID from selected trials, remove duplicates, ensure single value.

#==== Select Tracked Positions =======
recording = slice_stream(stream = streams['dlc:movement'],                  # selectors{} here keeps movement data belonging to the specified session only.
                        selectors = {'session' : sessionID }, 
)

position = recording['position']                                            # ['position'] extracts positional data from the movement stream.

position = position.sel(individual='individual_0',                          # .sel() selects the specified individual from the positional data.
                        drop=True,                                          # drop=True removes the individual dimension from the resulting DataArray, but leaves other dims intact.
)


## Compute Behavioural Measurements

In [ ]:
##### Compute Behavioural Measurements Using `movement` ################################

#==== Calculate Body Speed ==============
body_position = position.sel(keypoint='body', drop=True)
linear_speed = movement.kinematics.compute_speed(body_position)    # Calculate before restricting the recording to trials.


#==== Calculate Allocentric Head Direction
allocentric_hd = movement.kinematics.compute_forward_vector_angle(
    data             = position,
    left_keypoint    = 'lear',
    right_keypoint   = 'rear',
    reference_vector = (0, -1),                                  # Image-up is the fixed external reference.
    camera_view      = 'top_down',
    in_degrees       = False,                                    # Store angles in radians, as in the plotting notebooks.
)


#==== Create a Labelled Measurement Table =
features = pd.DataFrame({
    'body_x_px': body_position.sel(space='x').values,
    'body_y_px': body_position.sel(space='y').values,
    'linear_speed_px_s': linear_speed.values,
    'head_direction_rad': allocentric_hd.values,
}, index=position.time.values)
features.index.name = 'time_s'


## Convert to Pynapple and Select Trials

Convert the labelled measurements directly to a `TsdFrame`, then restrict it using trial intervals. Keep `selected_trials` for phase and event annotations: a Pynapple `IntervalSet` is time support, not a replacement for the trial table or its exact event/inclusion rules. This cell uses **Pynapple's** interval selection.

References: [TsdFrame](https://pynapple.org/generated/pynapple.TsdFrame.html), [interval restriction](https://pynapple.org/user_guide/03_core_methods.html).

In [ ]:
##### Convert the Measurements to Pynapple #############################################

#==== Keep the Column Labels and Clock ==
behaviour = nap.TsdFrame(
    t          = features.index.to_numpy(),
    d          = features.to_numpy(),
    columns    = features.columns,
    time_units = 's',                                            # Do not shift or rescale the aligned recording clock.
)


#==== Restrict to the Selected Trials ====
selected_behaviour = behaviour                                  # None keeps the complete recording.
if selected_trials is not None:
    trial_epochs = nap.IntervalSet(
        start      = selected_trials['start_time'].to_numpy(),
        end        = selected_trials['end_time'].to_numpy(),
        time_units = 's',
    )
    selected_behaviour = behaviour.restrict(trial_epochs)

display(selected_behaviour)


## Save and Reload the Pynapple Object

In [ ]:
##### Save and Reload Using Pynapple's NPZ Format #######################################

#==== Save Measurements and Trial Labels =
selected_behaviour.save('selected_behaviour.npz')                 # Pynapple saves timestamps, values, column names and time support.
if selected_trials is not None:
    selected_trials.to_csv('selected_behaviour_trials.csv', index=False)


#==== Reload the Same Object ============
reloaded_behaviour = nap.load_file('selected_behaviour.npz')


## Pynapple Time-Series Plotting Function

In [ ]:
def add_annotations(ax,
                    selected_trials,
                    **kwargs
) -> AnnotationResult:
    """Add trial annotations to an axis using recording seconds.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axis with its time limits already set.
    selected_trials : pandas.DataFrame
        Trial rows from the recording being plotted.
    **kwargs
        Arguments passed to annotate_qc_timeseries.

    Returns
    -------
    AnnotationResult
        The annotation artists, legend handles and notes.
    """
    return annotate_qc_timeseries(ax=ax, trials=selected_trials, **kwargs)


In [ ]:
##### Plot a Pynapple Time Series on Ax #################################################


def plot_ax_pynapple_timeseries(
    ax: plt.Axes,
    series: nap.Tsd,
    selected_trials: pd.DataFrame | None = None,
    ylabel: str = 'Value',
    row_duration: float | None = 60.0,
    subtitle: str | None = 'Pynapple Time Series',
    add_annotations_func: Callable[..., AnnotationResult] | None = add_annotations,
    annotation_kwargs: dict | None = None,
    **kwargs,
) -> plt.Axes:
    """Plot a selected Pynapple measurement in consecutive recording-time rows.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axis reserved for the plot. The caller sets figure size.
    series : pynapple.Tsd
        Nonempty one-dimensional measurement with ordered time in seconds.
    selected_trials : pandas.DataFrame or None
        Optional trial rows used only for annotations.
    ylabel : str
        Measurement name and its units.
    row_duration : float or None
        Recording seconds per row. None uses one row for the complete range.
    subtitle : str or None
        Title above the first row. None omits it.
    add_annotations_func : callable or None
        Default helper uses selected_trials and is skipped without trials.
        Custom callbacks receive only the axis and annotation_kwargs.
    annotation_kwargs : dict or None
        Callback settings. Supply default-helper trials via selected_trials.
    **kwargs
        Arguments passed to Axes.plot, such as markersize and alpha.

    Returns
    -------
    matplotlib.axes.Axes
        Original axis; multiple rows are available through ax.child_axes.
    """
    #==== 1| Calculate Rows Needed for Subplot =======================
    range_start = float(series.t[0])
    range_end = float(series.t[-1])

    duration = row_duration
    if duration is None:
        duration = (range_end - range_start) or 1.0                # Give a lone sample a one-second row.
    row_count = max(1, int(np.ceil((range_end - range_start) / duration)))


    #==== 2| Create the Required Time Rows ===========================
    if row_count == 1:
        row_axes = [ax]
    else:
        ax.set_axis_off()                                        # Use the supplied axis as the row container.
        row_axes = []
        row_height = 1 / (row_count + 0.25 * (row_count - 1))
        row_gap = 0.25 * row_height                              # Leave room for each row's time labels.

        for row in range(row_count):
            bottom = 1 - (row + 1) * row_height - row * row_gap
            row_axes.append(ax.inset_axes([0, bottom, 1, row_height]))

    for row_ax in row_axes[1:]:
        row_ax.sharey(row_axes[0])                               # Keep the measurement scale the same across rows.


    #==== 3| Plot the Measurement and Matching Annotations ===========
    annotation_kwargs = annotation_kwargs or {}

    for row, row_ax in enumerate(row_axes):
        row_start = range_start + row * duration
        row_end = min(row_start + duration, range_end)
        in_row = (series.t >= row_start) & (series.t <= row_end)

        row_ax.plot(series.t[in_row], series.values[in_row], '.', **kwargs)
        row_ax.set(xlim=(row_start, row_start + duration),
                   xlabel='Time (Seconds)', ylabel=ylabel)

        # Keep annotation timestamps on the same recording clock as Pynapple.
        if add_annotations_func is add_annotations:
            if selected_trials is not None:
                add_annotations_func(row_ax, selected_trials, **annotation_kwargs)
        elif add_annotations_func is not None:
            add_annotations_func(row_ax, **annotation_kwargs)


    #==== 4| Set the Shared Scale and Add the Plot Title ==============
    row_axes[0].autoscale(axis='y')                               # Include every row after annotations restore their old limits.
    if subtitle is not None:
        row_axes[0].set_title(subtitle)

    return ax


## Generate Plot

In [ ]:
##### Plot the Round-Tripped Measurement ###############################################

#==== Choose a Labelled Measurement ======
series = reloaded_behaviour['linear_speed_px_s']
row_duration = 60.0
plot_duration = float(series.t[-1] - series.t[0])
row_count = max(1, int(np.ceil(plot_duration / row_duration)))
fig, ax = plt.subplots(figsize=(13, 2.8 * row_count), layout='constrained')


#==== Add the Original Trial Annotations =
ax = plot_ax_pynapple_timeseries(
    ax                   = ax,
    series               = series,
    selected_trials      = selected_trials,
    ylabel               = 'Linear Speed (Pixels/Seconds)',
    row_duration         = row_duration,
    subtitle             = 'movement Speed | Pynapple NPZ Round Trip',
    add_annotations_func = add_annotations,
    annotation_kwargs    = None,
    markersize           = 2.5,
)

fig.savefig('pynapple_integration.svg', bbox_inches='tight')
plt.show()


This demonstrates conversion, interval selection and a behavioural NPZ round trip; it does not resynchronise clocks or assume a spike-file schema. An arbitrary NPZ is not automatically a Pynapple file.

Reference: [Pynapple input/output](https://pynapple.org/user_guide/02_input_output.html).